#Quality Checks
#####Script Purpose
###### This script performs quality checks to validate the integrity, consistency, and accuracy of the Gold Layer. These checks ensure:
- Uniqueness of surrogate keys in dimension tables.
- Referential integrity between fact and dimension tables.
- Validation of relationships in the data model for analytical purposes.
	
#####Usage Notes:
- Investigate and resolve any discrepancies found during the checks.

#'gold.dim_customers'

##Checking after joining tables from Silver Layer

####Check for duplicates
- Expectation: No results

In [0]:
query = """
SELECT customer_id, COUNT(*) FROM (
    SELECT
        ci.customer_id,
        ci.customer_number,
        ci.first_name,
        ci.last_name,
        ci.marital_status,
        ci.gender,
        ci.created_date,
        ca.birth_date,
        ca.gender,
        la.country
    FROM silver.crm_customers AS ci
    LEFT JOIN silver.erp_customers AS ca
        ON ci.customer_number = ca.customer_number
    LEFT JOIN silver.erp_customer_location AS la
        ON ci.customer_number = la.customer_number
) AS t GROUP BY customer_id
HAVING COUNT(*) > 1
"""
df = spark.sql(query)

df.display()

customer_id,COUNT(*)


####Checking differences between 'ci.gender' and 'ca.gender' for Data Integration

In [0]:
query = """
SELECT DISTINCT
    ci.gender,
    ca.gender,
    CASE 
        WHEN ci.gender != 'n/a' THEN ci.gender         -- CRM is the primary source for gender
        ELSE COALESCE(ca.gender, 'n/a')  			   -- Fallback to ERP data
    END AS new_gen
FROM silver.crm_customers AS ci
LEFT JOIN silver.erp_customers AS ca
    ON ci.customer_number = ca.customer_number
LEFT JOIN silver.erp_customer_location AS la
    ON ci.customer_number = la.customer_number
ORDER BY 1,2
"""
df = spark.sql(query)

df.display()

gender,gender,new_gen
Female,Female,Female
Female,Male,Female
Female,n/a,Female
Male,Female,Male
Male,Male,Male
Male,n/a,Male
n/a,Female,Female
n/a,Male,Male
n/a,n/a,n/a


##Checking 'gold.dim_customers'

####Check the 'gold.dim_customers'

In [0]:
query = """
SELECT * FROM gold.dim_customers
"""
df = spark.sql(query)

df.limit(10).display()

customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date
1,11000,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
2,11001,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
3,11002,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
4,11003,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
5,11004,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
6,11005,AW00011005,Julio,Ruiz,Australia,Single,Male,1976-08-01,2025-10-06
7,11006,AW00011006,Janet,Alvarez,Australia,Single,Female,1976-12-02,2025-10-06
8,11007,AW00011007,Marco,Mehta,Australia,Married,Male,1969-11-06,2025-10-06
9,11008,AW00011008,Rob,Verhoff,Australia,Single,Female,1975-07-04,2025-10-06
10,11009,AW00011009,Shannon,Carlson,Australia,Single,Male,1969-09-29,2025-10-06


In [0]:
query = """
SELECT DISTINCT gender FROM gold.dim_customers
"""
df = spark.sql(query)

df.display()

gender
Male
Female
n/a


####Check for Uniqueness of Customer Key in gold.dim_customers
- Expectation: No results

In [0]:
query = """
SELECT 
    customer_id,
    COUNT(*) AS duplicate_count
FROM gold.dim_customers
GROUP BY customer_id
HAVING COUNT(*) > 1
"""
df = spark.sql(query)

df.display()

customer_id,duplicate_count


#'gold.dim_products'

####Checking uniqueness
- Expectation: No results

In [0]:
query = """
SELECT product_number, COUNT(*) FROM (
    SELECT
        pn.product_id,
        pn.category_id,
        pn.product_number,
        pn.product_name,
        pn.product_cost,
        pn.product_line,
        pn.start_date,
        pn.end_date,
        pc.category,
        pc.subcategory,
        pc.maintenance_flag
    FROM silver.crm_products AS pn
    LEFT JOIN silver.erp_product_category AS pc
        ON pn.category_id = pc.category_id
    WHERE end_date IS NULL                -- Filter out all historical data
) AS t GROUP BY product_number
HAVING COUNT(*) > 1
"""
df = spark.sql(query)

df.display()

product_number,COUNT(*)


####Checking 'gold.dim_products'

In [0]:
query = """
SELECT * FROM gold.dim_products
"""
df = spark.sql(query)

df.limit(10).display()

product_key,product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_cost,product_line,start_date
1,210,FR-R92B-58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,true,0,Road,2003-07-01
2,211,FR-R92R-58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,true,0,Road,2003-07-01
3,348,BK-M82B-38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
4,349,BK-M82B-42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
5,350,BK-M82B-44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
6,351,BK-M82B-48,Mountain-100 Black- 48,BI_MB,Bikes,Mountain Bikes,true,1898,Mountain,2011-07-01
7,344,BK-M82S-38,Mountain-100 Silver- 38,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
8,345,BK-M82S-42,Mountain-100 Silver- 42,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
9,346,BK-M82S-44,Mountain-100 Silver- 44,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01
10,347,BK-M82S-48,Mountain-100 Silver- 48,BI_MB,Bikes,Mountain Bikes,true,1912,Mountain,2011-07-01


####Checking 'gold.product_key'

####Check for Uniqueness of Product Key in gold.dim_products
- Expectation: No results 

In [0]:
query = """
SELECT 
    product_key,
    COUNT(*) AS duplicate_count
FROM gold.dim_products
GROUP BY product_key
HAVING COUNT(*) > 1
"""
df = spark.sql(query)

df.display()

product_key,duplicate_count


#'gold.fact_sales'

####Checking 'gold.fact_sales'

In [0]:
query = """
SELECT * FROM gold.fact_sales
"""
df = spark.sql(query)

df.limit(10).display()

order_number,product_key,customer_key,order_date,ship_date,due_date,sales_amount,quantity,price
SO43697,20,10769,2010-12-29,2011-01-05,2011-01-10,3578,1,3578.0
SO43698,9,17390,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43699,9,14864,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43700,41,3502,2010-12-29,2011-01-05,2011-01-10,699,1,699.0
SO43701,9,4,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43702,16,16646,2010-12-30,2011-01-06,2011-01-11,3578,1,3578.0
SO43703,20,5625,2010-12-30,2011-01-06,2011-01-11,3578,1,3578.0
SO43704,6,6,2010-12-30,2011-01-06,2011-01-11,3375,1,3375.0
SO43705,7,12,2010-12-30,2011-01-06,2011-01-11,3400,1,3400.0
SO43706,17,16622,2010-12-31,2011-01-07,2011-01-12,3578,1,3578.0


####Check the data model connectivity between fact and dimensions
#####Foreign Key Integrity (Dimensions)
- Expectation: No results

In [0]:
query = """
SELECT * 
FROM gold.fact_sales AS f
LEFT JOIN gold.dim_customers AS c
ON c.customer_key = f.customer_key
LEFT JOIN gold.dim_products AS p
ON p.product_key = f.product_key
WHERE p.product_key IS NULL OR c.customer_key IS NULL
"""
df = spark.sql(query)

df.display()

order_number,product_key,customer_key,order_date,ship_date,due_date,sales_amount,quantity,price,customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,create_date,product_key,product_id,product_number,product_name,category_id,category,subcategory,maintenance_flag,product_cost,product_line,start_date
